# Raw cross-run school drop-off check

This notebook reads the raw `tour.csv` and `trip_linked.csv` inputs for the four runs in `configs/will_config.yaml` and shows the cross-run school-tour and school-trip drop-off directly from the raw files.

In [1]:
from pathlib import Path

import polars as pl
from IPython.display import display


RUNS = {
    "Unfiltered": Path(r"C:\Users\wesley.darling\Downloads\viz\viz\unfiltered"),
    "Filtered": Path(r"C:\Users\wesley.darling\Downloads\viz\viz\filtered"),
    "Override": Path(r"C:\Users\wesley.darling\Downloads\viz\viz\override"),
    "Estimation Output": Path(r"C:\Users\wesley.darling\Downloads\viz\viz\output"),
}

SCHOOL_ALIASES = {"school", "univ", "university", "college"}


def read_raw_csv(path: Path) -> pl.DataFrame:
    return pl.read_csv(path, infer_schema_length=0)


def _is_non_numeric_text(series: pl.Series) -> bool:
    values = [str(v).strip() for v in series.drop_nulls().head(50).to_list()]
    if not values:
        return False
    return any(not value.lstrip('-').isdigit() for value in values)


def choose_purpose_expr(df: pl.DataFrame, candidates: list[str], alias: str) -> pl.Expr:
    available = [col for col in candidates if col in df.columns]
    if not available:
        return pl.lit(None, dtype=pl.Utf8).alias(alias)

    non_numeric = [col for col in available if _is_non_numeric_text(df[col].cast(pl.Utf8))]
    preferred = non_numeric if non_numeric else available
    return pl.coalesce([pl.col(col).cast(pl.Utf8) for col in preferred]).alias(alias)


def normalize_text(col: str) -> pl.Expr:
    return pl.col(col).cast(pl.Utf8).str.strip_chars().str.to_lowercase()


tour_rows = []
trip_rows = []

for run_label, run_dir in RUNS.items():
    tours = read_raw_csv(run_dir / "tour.csv")
    trips = read_raw_csv(run_dir / "trip_linked.csv")

    tours = tours.with_columns(
        choose_purpose_expr(
            tours,
            ["tour_purpose", "primary_purpose", "tour_type", "purpose"],
            "derived_tour_purpose",
        )
    )
    trips = trips.with_columns(
        choose_purpose_expr(
            trips,
            ["tour_purpose", "primary_purpose", "tour_type"],
            "derived_tour_purpose",
        ),
        choose_purpose_expr(
            trips,
            ["trip_purpose", "purpose"],
            "derived_trip_purpose",
        ),
    )

    tours = tours.with_columns(normalize_text("derived_tour_purpose").alias("tour_purpose_norm"))
    trips = trips.with_columns(
        normalize_text("derived_tour_purpose").alias("tour_purpose_norm"),
        normalize_text("derived_trip_purpose").alias("trip_purpose_norm"),
    )

    tour_total = tours.height
    trip_total = trips.height

    raw_school_tours = tours.filter(pl.col("tour_purpose_norm") == "school").height
    raw_univ_tours = tours.filter(pl.col("tour_purpose_norm").is_in(["univ", "university", "college"])).height
    grouped_school_tours = tours.filter(pl.col("tour_purpose_norm").is_in(sorted(SCHOOL_ALIASES))).height

    raw_school_trip_rows = trips.filter(pl.col("tour_purpose_norm") == "school").height
    raw_univ_trip_rows = trips.filter(pl.col("tour_purpose_norm").is_in(["univ", "university", "college"])).height
    grouped_school_trip_rows = trips.filter(pl.col("tour_purpose_norm").is_in(sorted(SCHOOL_ALIASES))).height
    school_destination_trip_rows = trips.filter(pl.col("trip_purpose_norm") == "school").height

    tour_rows.append(
        {
            "run": run_label,
            "all_tours": tour_total,
            "raw_school_tours": raw_school_tours,
            "raw_univ_tours": raw_univ_tours,
            "grouped_school_tours": grouped_school_tours,
            "grouped_school_tour_pct": grouped_school_tours / tour_total,
        }
    )
    trip_rows.append(
        {
            "run": run_label,
            "all_trip_rows": trip_total,
            "school_trip_rows_by_tour_purpose": raw_school_trip_rows,
            "univ_trip_rows_by_tour_purpose": raw_univ_trip_rows,
            "grouped_school_trip_rows_by_tour_purpose": grouped_school_trip_rows,
            "school_trip_rows_by_trip_purpose": school_destination_trip_rows,
            "grouped_school_trip_row_pct": grouped_school_trip_rows / trip_total,
        }
    )

tour_summary = pl.DataFrame(tour_rows)
trip_summary = pl.DataFrame(trip_rows)


In [2]:
display(tour_summary)
display(trip_summary)

run,all_tours,raw_school_tours,raw_univ_tours,grouped_school_tours,grouped_school_tour_pct
str,i64,i64,i64,i64,f64
"""Unfiltered""",218417,13386,0,13386,0.061286
"""Filtered""",72337,460,0,460,0.006359
"""Override""",72337,460,0,460,0.006359
"""Estimation Output""",72311,257,179,436,0.00603


run,all_trip_rows,school_trip_rows_by_tour_purpose,univ_trip_rows_by_tour_purpose,grouped_school_trip_rows_by_tour_purpose,school_trip_rows_by_trip_purpose,grouped_school_trip_row_pct
str,i64,i64,i64,i64,i64,f64
"""Unfiltered""",666745,31447,0,31447,14898,0.047165
"""Filtered""",207266,1202,0,1202,680,0.005799
"""Override""",207266,1202,0,1202,680,0.005799
"""Estimation Output""",207207,606,544,1150,656,0.00555


In [3]:
tour_summary.select(
    "run",
    "grouped_school_tours",
    (pl.col("grouped_school_tour_pct") * 100).round(3).alias("grouped_school_tour_pct"),
).sort("grouped_school_tours", descending=True)

run,grouped_school_tours,grouped_school_tour_pct
str,i64,f64
"""Unfiltered""",13386,6.129
"""Filtered""",460,0.636
"""Override""",460,0.636
"""Estimation Output""",436,0.603


In [4]:
trip_summary.select(
    "run",
    "grouped_school_trip_rows_by_tour_purpose",
    "school_trip_rows_by_trip_purpose",
    (pl.col("grouped_school_trip_row_pct") * 100).round(3).alias("grouped_school_trip_row_pct"),
).sort("grouped_school_trip_rows_by_tour_purpose", descending=True)

run,grouped_school_trip_rows_by_tour_purpose,school_trip_rows_by_trip_purpose,grouped_school_trip_row_pct
str,i64,i64,f64
"""Unfiltered""",31447,14898,4.716
"""Filtered""",1202,680,0.58
"""Override""",1202,680,0.58
"""Estimation Output""",1150,656,0.555
